# Percepción de seguridad
Impacto de la percepción de inseguridad en el uso de ECOBICI por alcaldía
Análisis de datos panel (alcaldía x trimestre), separado por género.

Marco metodológico (ver también las fuentes académicas revisadas antes:
Baltagi "Econometric Analysis of Panel Data"; Wooldridge "Introductory
Econometrics" cap. 13-14; Torres-Reyna "Panel Data Analysis: Fixed and Random Effects using Stata", Princeton):

**log(viajes_it) = β1 * inseguro_pct_it + β2 * estaciones_activas_it + αi (efecto fijo de ALCALDÍA)+ γt (efecto fijo de TRIMESTRE)  + εit **

  - αi absorbe todo lo que es constante en el tiempo dentro de cada
    alcaldía (densidad, uso de suelo, cultura de movilidad...) y que de
   otro modo sesgaría el coeficiente de inseguridad (variable omitida).
   - γt absorbe choques comunes a toda la ciudad en cada trimestre
     (estacionalidad, estacionalidad del clima, eventos citywide).
   - Errores estándar agrupados (clustered) por alcaldía, como recomienda
     Torres-Reyna, para no subestimar los errores por autocorrelación
     dentro de cada alcaldía a lo largo del tiempo.
   - Variable dependiente en logaritmo: la distribución de viajes está
     fuertemente sesgada a la derecha (Iztapalapa vs. Milpa Alta), y
     Wooldridge recomienda log() en este tipo de variables económicas
     para estabilizar la varianza e interpretar los coeficientes como
     semi-elasticidades aproximadas. Sigue siendo una regresión CONTINUA     (OLS panel), no un modelo de conteo.

# Generación del panel

In [ ]:
#   Ejercicio 1: sin inversión pública.
#   Ejercicio 2: se agrega log(inversión_pública) como control adicional.
#   Ambos ejercicios se corren por separado para mujeres (F) y hombres (M).

# PASO 0. Librerías
# -----------------------------------------------------------------------
!pip install linearmodels -q

import os
import unicodedata
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS, RandomEffects, compare

from google.colab import drive
drive.mount('/content/drive')

# -----------------------------------------------------------------------
# PASO 1. Rutas de tus 3 archivos en Drive (ajusta si están en otra carpeta)
# -----------------------------------------------------------------------

RUTA_SEGURIDAD = "/content/drive/MyDrive/Seguridad/percepcion_seguridad_alcaldias_cdmx.csv"
RUTA_ECOBICI   = "/content/drive/MyDrive/Tesis_caso/ecobici_resumen_viajes_trimestre_alcaldia.csv"
RUTA_INVERSION = "/content/drive/MyDrive/Seguridad/inversion_publica_alcaldia.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# -----------------------------------------------------------------------
# PASO 2. Normalización de nombres de alcaldía (para que las 3 bases
#          crucen bien, igual que en los scripts anteriores)
# -----------------------------------------------------------------------

def normalizar(texto):
    """
    Normaliza nombres para poder hacer merges entre distintas fuentes.

    Ejemplos:
        'Álvaro Obregón'  -> 'ALVARO OBREGON'
        'Benito Juárez'   -> 'BENITO JUAREZ'
        'Cuauhtémoc'      -> 'CUAUHTEMOC'
        'Miguel Hidalgo ' -> 'MIGUEL HIDALGO'
    """

    if pd.isna(texto):
        return None

    texto = str(texto).strip().upper()

    # Quitar acentos
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        caracter
        for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )

    # Eliminar espacios dobles
    texto = " ".join(texto.split())

    return texto


# =============================================================================
# ALCALDÍAS QUE FORMAN PARTE DEL ANÁLISIS DE ECOBICI
# =============================================================================

ALCALDIAS_ECOBICI = [
    "ALVARO OBREGON",
    "AZCAPOTZALCO",
    "BENITO JUAREZ",
    "COYOACAN",
    "CUAUHTEMOC",
    "MIGUEL HIDALGO"
]

def extraer_trimestre_numero(valor):
    """
    Soporta que la columna 'trimestre' venga como entero (1,2,3,4) o como
    texto ('T1','T2'...): siempre devuelve el número de trimestre (int).
    """
    digitos = "".join(c for c in str(valor) if c.isdigit())
    return int(digitos) if digitos else None


def detectar_columna(df, palabras_clave):
    """
    Encuentra la columna cuyo nombre contiene TODAS las palabras clave
    dadas (insensible a mayúsculas). Útil porque no conocemos el nombre
    exacto y completo de la columna de 'estaciones activas' en tu CSV.
    """
    for col in df.columns:
        low = col.lower()
        if all(palabra in low for palabra in palabras_clave):
            return col
    return None



In [ ]:
# PASO 3. Cargar y limpiar cada una de las 3 bases
# -----------------------------------------------------------------------
# --- 3.1 Seguridad ---------------------------------------------------
df_seguridad = pd.read_csv(RUTA_SEGURIDAD)
df_seguridad["alcaldia_norm"] = df_seguridad["alcaldia"].map(normalizar)
df_seguridad["trimestre"] = df_seguridad["trimestre"].map(extraer_trimestre_numero)
df_seguridad = df_seguridad[["anio", "trimestre", "alcaldia_norm", "inseguro_pct", "seguro_pct"]]

print(f"Seguridad: {len(df_seguridad)} filas, "
      f"{df_seguridad['anio'].nunique()} años, "
      f"{df_seguridad['alcaldia_norm'].nunique()} alcaldías.")

# --- 3.2 Ecobici --------------------------------------------------------
df_ecobici_raw = pd.read_csv(RUTA_ECOBICI)
df_ecobici_raw["alcaldia_norm"] = df_ecobici_raw["alcaldia"].map(normalizar)
df_ecobici_raw["trimestre"] = df_ecobici_raw["trimestre"].map(extraer_trimestre_numero)

# Normalizamos 'genero' (mayúsculas/espacios) y mostramos qué valores trae
# de verdad el CSV: si aparece algo distinto a F/M (p. ej. 'Otro', vacíos,
# o variantes de mayúsculas que antes de normalizar contarían como
# categorías separadas), el análisis F/M de abajo las ignora a propósito,
# pero es mejor que lo veas explícitamente en vez de que se pierdan solas.
df_ecobici_raw["genero"] = df_ecobici_raw["genero"].astype(str).str.strip().str.upper()
print("Valores únicos en 'genero' (ya normalizados):", df_ecobici_raw["genero"].unique())

col_estaciones = detectar_columna(df_ecobici_raw, ["estacion", "activ"])
if col_estaciones is None:
    raise KeyError(
        "No encontré una columna de 'estaciones activas' en el CSV de Ecobici. "
        f"Columnas disponibles: {list(df_ecobici_raw.columns)}. "
        "Ajusta `detectar_columna` o usa el nombre exacto a mano."
    )
print(f"Columna de estaciones activas detectada: '{col_estaciones}'")

# El CSV trae una fila por cada estación de retiro y género, así que:
#   - n_viajes se SUMA (total de viajes en esa alcaldía-trimestre-género)
#   - estaciones_activas NO se suma (es una propiedad de la alcaldía en ese
#     trimestre, repetida en cada fila): se toma su valor máximo/único.
viajes_agg = (
    df_ecobici_raw
    .groupby(["anio", "trimestre", "alcaldia_norm", "genero"], as_index=False)
    ["n_viajes"].sum()
)

estaciones_agg = (
    df_ecobici_raw
    .groupby(["anio", "trimestre", "alcaldia_norm"], as_index=False)
    [col_estaciones].max()
    .rename(columns={col_estaciones: "estaciones_activas"})
)

df_ecobici = viajes_agg.merge(
    estaciones_agg, on=["anio", "trimestre", "alcaldia_norm"], how="left"
)

print(f"Ecobici (agregado): {len(df_ecobici)} filas "
      f"(alcaldía x trimestre x género).")

# --- 3.3 Inversión pública (anual) --------------------------------------
df_inversion = pd.read_csv(RUTA_INVERSION)
# admite que la columna de año se llame 'año' o 'anio'
col_anio_inv = "año" if "año" in df_inversion.columns else "anio"
df_inversion = df_inversion.rename(columns={col_anio_inv: "anio"})
df_inversion["alcaldia_norm"] = df_inversion["alcaldia"].map(normalizar)
df_inversion = df_inversion[["anio", "alcaldia_norm", "inversion_publica"]]

print(f"Inversión pública: {len(df_inversion)} filas, "
      f"{df_inversion['anio'].nunique()} años.")



Seguridad: 400 filas, 7 años, 16 alcaldías.
Valores únicos en 'genero' (ya normalizados): ['F' 'M' 'O' '?']
Columna de estaciones activas detectada: 'estaciones_activas_alcaldia'
Ecobici (agregado): 450 filas (alcaldía x trimestre x género).
Inversión pública: 96 filas, 6 años.


In [ ]:
#viajes_agg,  df_inversion, estaciones
#viajes_agg.head(20)
#df_inversion.head(20)
#Después del diagnóstico, sabemos que el segundo trimestre ya no tiene datos (de 2026)

#Revisión de descartes de datos

La opción A:
A. 1 Las estaciones marcadas como 'SIN_CATÁLOGO' se deben  a un error de registro en la BD de ecobici, pues ocupa dos nombres o dos idetificadores y el código no está diseñado para clasificarlas como parte de alguna alcaldía. Estpa condición es mínima por lo que se decide decartarlo.

A.2 Cuauhtémoc en 2010, Miguel Hidalgo en 2012, Benito Juárez en 2015, Coyoacán en febrero de 2023, y Álvaro Obregón y Azcapotzalco en noviembre de 2023. (Por ello está bien que no aparezcan en la base de datos en ese momento) Iztapalapa, Iztacalco y Tlalpan solo están anunciadas para 2026.

B. Se refiere a los datos que no están registrados en la ENSU.

In [ ]:
# =============================================================================
# DE ENSU CONSERVAR SOLO LAS ALCALDÍAS CON ECOBICI
# =============================================================================

df_seguridad = df_seguridad[
    df_seguridad["alcaldia_norm"].isin(ALCALDIAS_ECOBICI)
].copy()

df_seguridad = df_seguridad.reset_index(drop=True)

print("Alcaldías en ECOBICI:")
print(sorted(df_ecobici["alcaldia_norm"].dropna().unique()))

print("\nAlcaldías en ENSU después del filtro:")
print(sorted(df_seguridad["alcaldia_norm"].dropna().unique()))

Alcaldías en ECOBICI:
['ALVARO OBREGON', 'AZCAPOTZALCO', 'BENITO JUAREZ', 'COYOACAN', 'CUAUHTEMOC', 'MIGUEL HIDALGO', 'SIN_CATALOGO']

Alcaldías en ENSU después del filtro:
['ALVARO OBREGON', 'AZCAPOTZALCO', 'BENITO JUAREZ', 'COYOACAN', 'CUAUHTEMOC', 'MIGUEL HIDALGO']


In [ ]:
panel = df_ecobici.merge(
    df_seguridad, on=["anio", "trimestre", "alcaldia_norm"], how="inner"
)
panel = panel.merge(
    df_inversion, on=["anio", "alcaldia_norm"], how="left"
)
# =============================================================================
# DIAGNÓSTICO: ¿qué filas se descartan y por qué?
#
# Hay DOS lugares donde se pierden filas:
#   A) El cruce (merge) con how="inner": las filas de Ecobici que no tienen
#      pareja en ENSU se pierden EN SILENCIO (no aparecen en ningún aviso).
#   B) El dropna: quita las filas donde falta n_viajes, inseguro_pct o
#      estaciones_activas. Esas son las 24 filas del aviso.
# =============================================================================

import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

LLAVES = ["anio", "trimestre", "alcaldia_norm"]
COLS_CLAVE = ["n_viajes", "inseguro_pct", "estaciones_activas"]


# -----------------------------------------------------------------------
# PARTE A. Filas que se pierden en el merge (inner) y nunca avisan
# -----------------------------------------------------------------------
auditoria_merge = df_ecobici.merge(
    df_seguridad[LLAVES + ["inseguro_pct"]],
    on=LLAVES, how="outer", indicator=True,
)

solo_ecobici = auditoria_merge[auditoria_merge["_merge"] == "left_only"]
solo_ensu = auditoria_merge[auditoria_merge["_merge"] == "right_only"]

print("=" * 70)
print("A) MERGE ECOBICI + ENSU")
print("=" * 70)
print(f"Combinaciones alcaldía-trimestre con viajes de Ecobici pero SIN dato "
      f"de ENSU (se pierden en el merge): "
      f"{solo_ecobici[LLAVES].drop_duplicates().shape[0]}")


print(f"\nCombinaciones con dato de ENSU pero SIN viajes de Ecobici "
      f"(no entran al panel): {solo_ensu[LLAVES].drop_duplicates().shape[0]}")
print("(Es normal que aquí aparezcan las 10 alcaldías sin Ecobici; "
      "revisa que no falten alcaldías con Ecobici.)")

#Se retiran el drop_na porque recordemos que la encuesta de inseguridad solo tiene el primer trimestre de 2026.
# -----------------------------------------------------------------------
# PARTE B. Filas que quita el dropna, y qué columna les falta
# En la sección B descartamos los valores que no se aprobaron después
# -----------------------------------------------------------------------
mascara = panel[COLS_CLAVE].isna().any(axis=1)
descartadas = panel.loc[mascara].copy()

descartadas["columnas_faltantes"] = descartadas[COLS_CLAVE].isna().apply(
    lambda fila: ", ".join(c for c in COLS_CLAVE if fila[c]), axis=1
)

print("\n" + "=" * 70)
print(f"B) FILAS QUE QUITA EL DROP_NA: {len(descartadas)}")
print("=" * 70)

print("\n--- Por columna faltante (el 'por qué') ---")
print(descartadas["columnas_faltantes"].value_counts().to_string())

print("\n--- Por periodo ---")
print(descartadas.groupby(["anio", "trimestre"]).size().to_string())

print("\n--- Por alcaldía ---")
print(descartadas.groupby("alcaldia_norm").size().to_string())

print("\n--- Por género ---")
print(descartadas.groupby("genero").size().to_string())


A) MERGE ECOBICI + ENSU
Combinaciones alcaldía-trimestre con viajes de Ecobici pero SIN dato de ENSU (se pierden en el merge): 34

Combinaciones con dato de ENSU pero SIN viajes de Ecobici (no entran al panel): 39
(Es normal que aquí aparezcan las 10 alcaldías sin Ecobici; revisa que no falten alcaldías con Ecobici.)

B) FILAS QUE QUITA EL DROP_NA: 24

--- Por columna faltante (el 'por qué') ---
columnas_faltantes
inseguro_pct    24

--- Por periodo ---
anio  trimestre
2026  2            24

--- Por alcaldía ---
alcaldia_norm
ALVARO OBREGON    4
AZCAPOTZALCO      4
BENITO JUAREZ     4
COYOACAN          4
CUAUHTEMOC        4
MIGUEL HIDALGO    4

--- Por género ---
genero
?    6
F    6
M    6
O    6


In [ ]:
#Por alguna razón no hay datos de la ENSU para el segundo semestre. (Supongo que es por la pandemia)
#El INEGI canceló oficialmente el levantamiento de la Encuesta Nacional de Seguridad Pública Urbana (ENSU) correspondiente al segundo trimestre de 2020 debido a la
#emergencia sanitaria generada por el virus SARS-CoV-2 (COVID-19)
sub_df = df_seguridad[
    (df_seguridad["anio"] == 2020) &
    (df_seguridad["trimestre"] == 2)
]
display(sub_df)

,anio,trimestre,alcaldia_norm,inseguro_pct,seguro_pct


In [ ]:
# -----------------------------------------------------------------------
# PASO 4. Construir el panel maestro
# -----------------------------------------------------------------------
# NOTA METODOLÓGICA: la inversión pública es ANUAL, mientras que el resto
# del panel es TRIMESTRAL. Al hacer el merge por año, el mismo valor de
# inversión se repite en los 4 trimestres de ese año dentro de cada
# alcaldía.

panel = df_ecobici.merge(
    df_seguridad, on=["anio", "trimestre", "alcaldia_norm"], how="inner"
)
panel = panel.merge(
    df_inversion, on=["anio", "alcaldia_norm"], how="left"
)

# UTILIZAMOS EL % DE INSEGURIDAD
filas_antes = len(panel)
panel = panel.dropna(subset=["n_viajes", "inseguro_pct", "estaciones_activas"])
if len(panel) < filas_antes:
    print(f"[Aviso] Se descartaron {filas_antes - len(panel)} filas con "
          f"valores faltantes en viajes/inseguridad/estaciones.")

# Variable dependiente en log (ver justificación arriba)
panel["log_viajes"] = np.log(panel["n_viajes"] + 1)
# La inversión también suele venir muy sesgada (alcaldías grandes invierten
# mucho más en términos absolutos); se usa en log por la misma razón.
panel["log_inversion"] = np.log(panel["inversion_publica"].clip(lower=1))

print(f"\nPanel final: {len(panel)} observaciones "
      f"({panel['alcaldia_norm'].nunique()} alcaldías x "
      f"{panel[['anio','trimestre']].drop_duplicates().shape[0]} periodos x "
      f"{panel['genero'].nunique()} géneros).")


[Aviso] Se descartaron 24 filas con valores faltantes en viajes/inseguridad/estaciones.

Panel final: 345 observaciones (6 alcaldías x 24 periodos x 4 géneros).


In [ ]:
panel
panel.to_csv("/content/drive/MyDrive/Tesis_caso/panel_ecobici.csv",
    index=False,
    encoding="utf-8-sig")